# 02 · Extract from Lakebase

Pulls new and changed rows out of Lakebase and lands them as Parquet files, ready for the pipeline to pick up.

**What it does**
- Reads only the rows that changed since the last run, using each table's `updated_at` as a bookmark
- Writes each run to its own folder in the `lakebase_extracts` volume
- Saves the new bookmark per table in `lakebase_watermarks`
- To test incremental behaviour, run src/tools/simulate_order_update first, then this notebook

**Good to know**
- The first run has no bookmark, so it copies everything. After that, runs only carry what changed.
- Safe to re-run. If a run fails halfway, the next one may extract a few rows again; the pipeline dedupes them by key.

In [0]:
%pip install -U databricks-sdk "psycopg[binary]"

In [0]:
dbutils.library.restartPython()

In [0]:
# Incremental extract: Lakebase (OLTP) -> Parquet files in a UC volume.
# Each run pulls only rows whose updated_at is newer than the previous
# run's high-water mark, so re-running is cheap and never duplicates data.

from datetime import datetime, timezone
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient

spark.conf.set("spark.sql.session.timeZone", "UTC")   # all timestamps handled in UTC

CATALOG   = "de_databricks_bp_wave_5"
LANDING   = f"{CATALOG}.olist_dev_landing"
VOLUME    = "lakebase_extracts"
VOL_PATH  = f"/Volumes/{CATALOG}/olist_dev_landing/{VOLUME}"
WM_TABLE  = f"{LANDING}.lakebase_watermarks"

ENDPOINT  = "projects/olist-capstone/branches/production/endpoints/primary"
HOST      = "ep-noisy-grass-d1gvpml8.database.us-west-2.cloud.databricks.com"
DB        = "databricks_postgres"
PG_SCHEMA = "olist"
USER      = spark.sql("select current_user()").first()[0]

TABLES = ["customers", "sellers", "products", "orders",
          "order_items", "order_payments", "order_reviews"]

w = WorkspaceClient()

def pg_token() -> str:
    """Short-lived OAuth token (1h) used as the Postgres password."""
    return w.postgres.generate_database_credential(endpoint=ENDPOINT).token

In [0]:
# One-time setup (idempotent): landing volume + watermark bookkeeping table.

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {LANDING}.{VOLUME}
    COMMENT 'Incremental Parquet extracts from Lakebase, one folder per table'""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WM_TABLE} (
        table_name      STRING NOT NULL,
        last_updated_at TIMESTAMP,
        last_run_at     TIMESTAMP,
        rows_extracted  BIGINT)
    COMMENT 'High-water mark per Lakebase table for incremental extracts'""")

print("Volume and watermark table ready")

In [0]:
# Extract rows changed since the last run, land them as Parquet,
# then advance each table's high-water mark.

token  = pg_token()
url    = f"jdbc:postgresql://{HOST}:5432/{DB}?sslmode=require"
run_ts = datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC
run_id = run_ts.strftime("%Y%m%dT%H%M%SZ")

watermarks = {r.table_name: r.last_updated_at for r in spark.table(WM_TABLE).collect()}
results = []

for table in TABLES:
    wm = watermarks.get(table)
    where = f"where updated_at > timestamptz '{wm.isoformat()}+00:00'" if wm else ""
    query = f"select * from {PG_SCHEMA}.{table} {where}"

    df = (spark.read.format("jdbc")
            .option("url", url)
            .option("query", query)
            .option("user", USER)
            .option("password", token)
            .load()
            .withColumn("_extracted_at", F.lit(run_ts)))

    out = f"{VOL_PATH}/{table}/run_id={run_id}"
    df.write.mode("overwrite").parquet(out)

    # Stats come from the files actually written, so the watermark can never
    # move past rows that were not landed.
    stats  = spark.read.parquet(out).agg(F.count("*").alias("n"),
                                         F.max("updated_at").alias("max_ts")).first()
    new_wm = stats.max_ts or wm
    results.append((table, new_wm, run_ts, stats.n))
    print(f"{table:16s} {stats.n:>9,} rows  (previous mark: {wm})")

updates = spark.createDataFrame(
    results,
    "table_name string, last_updated_at timestamp, last_run_at timestamp, rows_extracted bigint")
updates.createOrReplaceTempView("wm_updates")
spark.sql(f"""
    MERGE INTO {WM_TABLE} t
    USING wm_updates s ON t.table_name = s.table_name
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *""")

print(f"\nRun {run_id} complete")

In [0]:
display(spark.table(WM_TABLE).orderBy("table_name"))